In [1]:
import sys, time
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier, IsolationForest
from sklearn.metrics import average_precision_score, precision_recall_curve

from src.data import load_transactions
from src.train import FEATURES, split_by_time

In [2]:
train_df, test_df = split_by_time(load_transactions())
X_train, y_train = train_df[FEATURES], train_df["Class"]
X_test, y_test = test_df[FEATURES], test_df["Class"]

def recall_at_precision(y, scores, min_p=0.8):
    p, r, _ = precision_recall_curve(y, scores)
    ok = p >= min_p
    return r[ok].max() if ok.any() else 0.0

def single_row_ms(score_fn, X, n=200):
    start = time.perf_counter()
    for i in range(n):
        score_fn(X.iloc[[i]])
    return (time.perf_counter() - start) / n * 1000

In [3]:
models = {
    "LogReg": Pipeline([
        ("scale", StandardScaler()),
        ("clf", LogisticRegression(class_weight="balanced", max_iter=1000)),
    ]),
    "RandomForest": RandomForestClassifier(
        n_estimators=100, max_depth=12, class_weight="balanced_subsample",
        n_jobs=-1, random_state=42),
    "HistGradBoost": HistGradientBoostingClassifier(
        class_weight="balanced", random_state=42),
}

rows = []
for name, m in models.items():
    m.fit(X_train, y_train)
    scores = m.predict_proba(X_test)[:, 1]
    rows.append({
        "model": name,
        "pr_auc": average_precision_score(y_test, scores),
        "recall@p80": recall_at_precision(y_test, scores),
        "ms_per_row": single_row_ms(lambda X: m.predict_proba(X), X_test),
    })
pd.DataFrame(rows).round(4)

,model,pr_auc,recall@p80,ms_per_row
0,LogReg,0.7622,0.7467,1.2378
1,RandomForest,0.8114,0.7733,17.6734
2,HistGradBoost,0.7355,0.7200,2.0394


In [4]:
iso = IsolationForest(n_estimators=100, contamination=0.0017, random_state=42, n_jobs=-1)
iso.fit(X_train)
iso_scores = -iso.score_samples(X_test)

print("PR-AUC    :", average_precision_score(y_test, iso_scores))
print("recall@p80:", recall_at_precision(y_test, iso_scores))

PR-AUC    : 0.025510889624925943
recall@p80: 0.0


In [5]:
from sklearn.metrics import precision_score, recall_score

df = load_transactions().sort_values("Time")
n = len(df)
train_df = df.iloc[: int(n * 0.6)]
val_df = df.iloc[int(n * 0.6): int(n * 0.8)]
test_df = df.iloc[int(n * 0.8):]
print(len(train_df), len(val_df), len(test_df))

170884 56961 56962


In [6]:
model = models["LogReg"]
model.fit(train_df[FEATURES], train_df["Class"])

val_scores = model.predict_proba(val_df[FEATURES])[:, 1]
p, r, t = precision_recall_curve(val_df["Class"], val_scores)

ok = np.where(p[:-1] >= 0.8)[0]
threshold = t[ok[0]] if len(ok) else 0.5
print("threshold:", threshold)

threshold: 0.9999561803866615


In [7]:
y_test = test_df["Class"]
test_scores = model.predict_proba(test_df[FEATURES])[:, 1]

for name, th in [("default 0.5", 0.5), ("tuned", threshold)]:
    preds = (test_scores >= th).astype(int)
    print(name, "| threshold", round(th, 4),
          "| precision", round(precision_score(y_test, preds, zero_division=0), 3),
          "| recall", round(recall_score(y_test, preds), 3),
          "| flagged", preds.sum())

default 0.5 | threshold 0.5 | precision 0.044 | recall 0.893 | flagged 1520
tuned | threshold 1.0 | precision 0.879 | recall 0.68 | flagged 58


In [8]:
import sys, json
from pathlib import Path
sys.path.append(str(Path.cwd().parent))
from src.data import load_transactions
from src.train import FEATURES

row = load_transactions().query("Class == 1").iloc[0][FEATURES].to_dict()
print(json.dumps(row))

{"V1": -2.3122265423263, "V2": 1.95199201064158, "V3": -1.60985073229769, "V4": 3.9979055875468, "V5": -0.522187864667764, "V6": -1.42654531920595, "V7": -2.53738730624579, "V8": 1.39165724829804, "V9": -2.77008927719433, "V10": -2.77227214465915, "V11": 3.20203320709635, "V12": -2.89990738849473, "V13": -0.595221881324605, "V14": -4.28925378244217, "V15": 0.389724120274487, "V16": -1.14074717980657, "V17": -2.83005567450437, "V18": -0.0168224681808257, "V19": 0.416955705037907, "V20": 0.126910559061474, "V21": 0.517232370861764, "V22": -0.0350493686052974, "V23": -0.465211076182388, "V24": 0.320198198514526, "V25": 0.0445191674731724, "V26": 0.177839798284401, "V27": 0.261145002567677, "V28": -0.143275874698919, "Amount": 0.0}
